In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Cargar imagen desde archivo
image = cv2.imread('img/IMG-20260318-WA0001.jpg')

# Verificar si la imagen se cargó correctamente
if image is None:
    print("Error: No se pudo cargar la imagen. Por favor, verifica la ruta y el formato del archivo.")
else:
    # Si la imagen se cargó en color, convertirla a escala de grises
    if len(image.shape) == 3:
        image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        image_gray = image # Ya está en escala de grises

    # Escalar la imagen a 96x96
    image_resized = cv2.resize(image_gray, (96, 96))

    # Guardar la imagen redimensionada en la carpeta img
    cv2.imwrite('img/image_resized_96x96.jpg', image_resized)

    # Aplicar el operador Sobel para detectar bordes en la dirección horizontal
    sobel_horizontal = cv2.Sobel(image_resized, cv2.CV_64F, 1, 0, ksize=3)

    # Aplicar el operador Sobel para detectar bordes en la dirección vertical
    sobel_vertical = cv2.Sobel(image_resized, cv2.CV_64F, 0, 1, ksize=3)

    # Mostrar las imágenes originales y los resultados del Sobel
    plt.figure(figsize=(10, 10))

    # Imagen original redimensionada
    plt.subplot(1, 3, 1)
    plt.title('Imagen Redimensionada')
    plt.imshow(image_resized, cmap='gray')
    plt.axis('off')

    # Resultado Sobel Horizontal
    plt.subplot(1, 3, 2)
    plt.title('Sobel Horizontal')
    plt.imshow(sobel_horizontal, cmap='gray')
    plt.axis('off')

    # Resultado Sobel Vertical
    plt.subplot(1, 3, 3)
    plt.title('Sobel Vertical')
    plt.imshow(sobel_vertical, cmap='gray')
    plt.axis('off')

    plt.show()

## Generar header C con los datos de la imagen para el ESP32

In [ ]:
import cv2
import numpy as np

# Cargar la imagen redimensionada
img = cv2.imread('img/image_resized_96x96.jpg', cv2.IMREAD_GRAYSCALE)

# Generar archivo header C
header_path = 'esp32/main/image_data.h'
with open(header_path, 'w') as f:
    f.write('#ifndef IMAGE_DATA_H\n')
    f.write('#define IMAGE_DATA_H\n\n')
    f.write('#include <stdint.h>\n\n')
    f.write(f'// Imagen en escala de grises {img.shape[1]}x{img.shape[0]}\n')
    f.write(f'static const uint8_t image_data[{img.shape[0]}][{img.shape[1]}] = {{\n')
    for y in range(img.shape[0]):
        row = ', '.join(str(img[y, x]) for x in range(img.shape[1]))
        f.write(f'    {{{row}}}')
        if y < img.shape[0] - 1:
            f.write(',')
        f.write('\n')
    f.write('};\n\n')
    f.write('#endif // IMAGE_DATA_H\n')

print(f'Header generado en: {header_path}')
print(f'Dimensiones: {img.shape[0]}x{img.shape[1]}')

## Google Colab: Leer salida serial del ESP32 y graficar resultados Sobel

Copiar la salida del monitor serial del ESP32 en la variable `serial_output` de la siguiente celda, o pegar el contenido en un archivo `serial_output.txt` y cargarlo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =============================================================
# Opcion 1: Pegar la salida del monitor serial directamente aqui
# =============================================================
serial_output = """
PEGAR AQUI LA SALIDA COMPLETA DEL MONITOR SERIAL DEL ESP32
"""

# =============================================================
# Opcion 2: Cargar desde archivo (descomentar si se usa archivo)
# =============================================================
# with open('serial_output.txt', 'r') as f:
#     serial_output = f.read()

# Parsear las matrices desde la salida serial
def parse_matrix(text, start_label, end_label):
    lines = text.split('\n')
    capturing = False
    rows = []
    for line in lines:
        line = line.strip()
        if line == start_label:
            capturing = True
            continue
        if line == end_label:
            capturing = False
            continue
        if capturing and line:
            row = [int(x) for x in line.split(',')]
            rows.append(row)
    return np.array(rows)

sobel_h = parse_matrix(serial_output, 'START_SOBEL_H', 'END_SOBEL_H')
sobel_v = parse_matrix(serial_output, 'START_SOBEL_V', 'END_SOBEL_V')

print(f'Sobel Horizontal: {sobel_h.shape}')
print(f'Sobel Vertical: {sobel_v.shape}')

# Graficar resultados
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.title('Sobel Horizontal (ESP32)')
plt.imshow(sobel_h, cmap='gray')
plt.axis('off')
plt.colorbar(shrink=0.8)

plt.subplot(1, 2, 2)
plt.title('Sobel Vertical (ESP32)')
plt.imshow(sobel_v, cmap='gray')
plt.axis('off')
plt.colorbar(shrink=0.8)

plt.tight_layout()
plt.show()